# EcoBuild | Reproducible Building Energy Prediction and Decision-Support Analysis

**Individual research notebook — Veronica Broadway**  
**Project:** EcoBuild, Building Energy Efficiency Predictor  
**Research focus:** data quality, exploratory analytics, predictive model evaluation, reliability across building configurations, and responsible decision support.

## Study roadmap
1. Document data provenance, methods, and environmental settings.
2. Audit the dataset and explore building design relationships.
3. Create a reproducible held-out evaluation with **both** heating load (Y1) and cooling load (Y2).
4. Compare a training-mean baseline, linear regression, random forest, and gradient boosting; select using training-only cross-validation.
5. Evaluate previously unseen holdout rows, analyze error by height, and inspect feature importance.
6. Compare ordinary versus geometry-grouped cross-validation, reproduce the exact 192-design grouping from Namita’s work, and quantify the effect using repeated holdouts.
7. Check subgroup reliability on new designs, attribute team SHAP and app-model contributions accurately, and export research-ready figures/tables.

**Scope statement:** This analysis evaluates predictions against **simulation-generated labels** from the UCI Energy Efficiency dataset. It does not establish performance on occupied buildings, cost savings, causal effects of design changes, or demographic fairness. Results printed here are generated when the notebook runs; narrative text is not a substitute for measured outputs.

**Research question (feasible for the current dataset):** How accurately do conventional and nonlinear regression models predict simulated heating and cooling loads from early-stage building characteristics, and how consistently do prediction errors vary across building-height configurations?

**Accessibility conventions:** Headings are hierarchical; no result relies on color alone. Figures include descriptive titles, explicit axis labels, readable text, and adjacent text alternatives. All essential chart results are also available in data tables. Colors are chosen for contrast on white; descriptions should be updated if figures change.

## 1. Reproducibility and project setup
This notebook is intended for **Google Colab** or a local Python installation with internet access for UCI retrieval. The only remote dependency is the public UCI dataset (ID 242). If retrieval fails, upload the authoritative ENB2012 Excel file; no synthetic replacement data are generated.

**Methodological decision:** Set the random seed, record software versions, and export all numerical results so claims in the paper can be traced to a run.

In [ ]:
# Lightweight setup: only install if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec("ucimlrepo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ucimlrepo"])

import os, json, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn import __version__ as sklearn_version

SEED = 42
OUT = Path("ecobuild_research_outputs")
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
                     "savefig.dpi": 250, "figure.dpi": 110})
FEATURES = [f"X{i}" for i in range(1, 9)]
TARGETS = ["Y1", "Y2"]
NAMES = {"X1":"Relative compactness", "X2":"Surface area", "X3":"Wall area",
         "X4":"Roof area", "X5":"Overall height", "X6":"Orientation",
         "X7":"Glazing area", "X8":"Glazing area distribution",
         "Y1":"Heating load", "Y2":"Cooling load"}
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit_learn": sklearn_version}
print("Software:", VERSIONS, "\nOutput folder:", OUT.resolve())

## 2. Dataset source, variable dictionary, and governance
**Source:** UCI Machine Learning Repository, *Energy Efficiency* dataset (ID 242), originally associated with Tsanas and Xifara (2012). The 768 rows represent simulated building configurations, not measured buildings. There are eight inputs (X1–X8) and two numerical outcomes (Y1 heating load; Y2 cooling load).

| Field | Description | Interpretation |
|---|---|---|
| X1 | Relative compactness | Design geometry |
| X2 | Surface area | Building envelope geometry |
| X3 | Wall area | Building envelope geometry |
| X4 | Roof area | Building envelope geometry |
| X5 | Overall height | Two design categories, 3.5 m and 7.0 m |
| X6 | Orientation | Encoded design direction; categorical, not a continuous physical scale |
| X7 | Glazing area | Fractional design variable |
| X8 | Glazing area distribution | Encoded glazing arrangement; categorical |
| Y1 | Heating load | Regression target |
| Y2 | Cooling load | Regression target |

**Governance:** Public simulation data contain no person-level attributes. Keep attribution and dataset citation in the paper, record preprocessing decisions, do not infer operational energy consumption or environmental impact without field validation.

**Reproducibility:** The following cell attempts UCI retrieval, then looks for a local Excel/CSV file. A user-supplied file is used only if the expected columns exist.

In [ ]:
DATA_SOURCE = None
try:
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=242)
    raw = pd.concat([repo.data.features, repo.data.targets], axis=1)
    DATA_SOURCE = "UCI Energy Efficiency repository, dataset 242 (ucimlrepo)"
except Exception as exc:
    print("UCI fetch unavailable:", str(exc)[:250])
    candidates = [Path("EcoBuild Data - ENB2012_data.xlsx"), Path("ENB2012_data.xlsx"),
                  Path("ENB2012_data.csv"), Path("/content/ENB2012_data.xlsx")]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError("Upload ENB2012_data.xlsx (or EcoBuild Data - ENB2012_data.xlsx) "
                                "and rerun this cell; no substitute data will be used.") from exc
    raw = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path)
    DATA_SOURCE = str(path)

# Normalize column names if the repository returns full feature labels.
raw.columns = [str(c).strip() for c in raw.columns]
if not set(FEATURES + TARGETS).issubset(raw.columns):
    reverse = {"Relative Compactness":"X1", "Surface Area":"X2", "Wall Area":"X3",
               "Roof Area":"X4", "Overall Height":"X5", "Orientation":"X6",
               "Glazing Area":"X7", "Glazing Area Distribution":"X8",
               "Heating Load":"Y1", "Cooling Load":"Y2"}
    raw = raw.rename(columns=reverse)
missing = sorted(set(FEATURES + TARGETS) - set(raw.columns))
if missing: raise ValueError(f"Missing expected variables: {missing}; got {list(raw.columns)}")
df = raw[FEATURES + TARGETS].copy()
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("Source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
display(df.head().rename(columns=NAMES))

## 3. Data quality and structural audit
Check size, numeric types, missingness, duplicates, unique values, and ranges **before** modeling. Duplicate features with conflicting targets would need investigation; rows representing valid repeated configurations should not be removed automatically. Summarize exactly what is discovered in the paper's Data section.

**Accessibility:** Printed tables have explicit column names and do not encode quality solely with colors.

In [ ]:
quality = pd.DataFrame({
    "variable": FEATURES + TARGETS,
    "meaning": [NAMES[c] for c in FEATURES + TARGETS],
    "dtype": [str(df[c].dtype) for c in FEATURES + TARGETS],
    "missing": [int(df[c].isna().sum()) for c in FEATURES + TARGETS],
    "unique_values": [int(df[c].nunique()) for c in FEATURES + TARGETS],
    "minimum": [df[c].min() for c in FEATURES + TARGETS],
    "maximum": [df[c].max() for c in FEATURES + TARGETS],
})
display(quality)
print("Rows:", len(df), "| Full duplicate rows:", int(df.duplicated().sum()),
      "| Duplicate input rows:", int(df.duplicated(FEATURES).sum()))
print("Conflicting outputs for identical input configurations:",
      int((df.groupby(FEATURES, dropna=False)[TARGETS].nunique(dropna=False) > 1).any(axis=1).sum()))
quality.to_csv(OUT / "table_01_data_quality.csv", index=False)
if df.isna().any().any():
    raise ValueError("Missing or nonnumeric values found. Inspect source; decide how to handle before training.")
if len(df) != 768: warnings.warn("Dataset row count differs from the expected 768. Verify the source.")
if not set(df["X5"].unique()).issubset({3.5, 7.0}):
    warnings.warn("Unexpected height values: update the height-group audit before running it.")

## 4. Exploratory data analysis (EDA): central tendencies and design groups
EDA characterizes the sample **without treating associations as causal**. Evaluate target distributions, height-group means and medians, and design variable relationships. These summaries are descriptive observations, not trained model performance.

**Questions:** Do the two targets vary broadly across configurations? How do the height groups differ? Are within-height correlations different from pooled correlations?

In [ ]:
summary = df[TARGETS].agg(["count", "mean", "std", "min", "median", "max"]).T
summary.index = [NAMES[c] for c in summary.index]
height_summary = df.groupby("X5", observed=True).agg(
    n=("Y1", "size"), heating_mean=("Y1", "mean"), heating_median=("Y1", "median"),
    heating_min=("Y1", "min"), heating_max=("Y1", "max"),
    cooling_mean=("Y2", "mean"), cooling_median=("Y2", "median"))
display(summary.round(3))
display(height_summary.round(3))
print("Overall heating/cooling correlation:", round(df["Y1"].corr(df["Y2"]), 4))
for h, g in df.groupby("X5"):
    print(f"At height {h:g} m: compactness/heating correlation {g['X1'].corr(g['Y1']):.3f}")
summary.to_csv(OUT / "table_02_target_descriptives.csv")
height_summary.to_csv(OUT / "table_03_height_group_descriptives.csv")

### Figure 1. Heating and cooling load distributions
Two separate histograms preserve axis clarity and avoid comparing different counts on one ambiguous plot.

**Figure text alternative (update after executing):** The two distributions show the number of simulated configurations at different heating-load and cooling-load values. Vertical dashed lines identify their means. Read the accompanying summary table for exact means, minima, and maxima. The spread indicates variability in simulated loads, not measured energy use.

In [ ]:
for target in TARGETS:
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.hist(df[target], bins=24, color="#286090", edgecolor="white", linewidth=.8)
    avg = df[target].mean()
    ax.axvline(avg, color="#963F00", linestyle="--", linewidth=2.2,
               label=f"Mean = {avg:.2f}")
    ax.set(title=f"Distribution of simulated {NAMES[target].lower()} across building designs",
           xlabel=f"{NAMES[target]} (dataset units)", ylabel="Number of configurations")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(OUT / f"figure_01_{target}_distribution.png", bbox_inches="tight")
    plt.show()
    print(f"Accessible summary, {target}: n={len(df)}, min={df[target].min():.2f}, "
          f"mean={avg:.2f}, median={df[target].median():.2f}, max={df[target].max():.2f}.")

### Figure 2. Height-group load comparison
Use a grouped bar chart with values printed directly on each bar, plus the exact-value table above.

**Text alternative:** The plot compares mean simulated heating and cooling loads for 3.5 m and 7.0 m configurations. It distinguishes targets by both color and labels; the bars are not evidence that increasing height alone causes the observed load change. Geometry and other design characteristics differ together.

In [ ]:
hs = df.groupby("X5")[TARGETS].mean().reindex([3.5, 7.0]); pos = np.arange(len(hs)); width=.35
fig, ax=plt.subplots(figsize=(8,4.6))
for shift, target, color in [(-width/2,"Y1","#285E91"),(width/2,"Y2","#A14900")]:
    vals=hs[target].to_numpy(); bars=ax.bar(pos+shift,vals,width,label=NAMES[target],color=color)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=10)
ax.set_xticks(pos, ["Shorter (3.5 m)","Taller (7.0 m)"])
ax.set(title="Taller configurations have higher mean simulated thermal loads",
       xlabel="Overall building height",ylabel="Mean load (dataset units)")
ax.legend(); ax.set_ylim(0,hs.to_numpy().max()*1.2);fig.tight_layout()
fig.savefig(OUT / "figure_02_load_by_height.png",bbox_inches="tight");plt.show()
print("Mean heating ratio, taller / shorter:",round(hs.loc[7.0,"Y1"]/hs.loc[3.5,"Y1"],3))

### Figure 3. Compactness and heating load, separated by building height
**Why stratify?** Pooled trends may obscure relationships within design subgroups. Both height and compactness reflect building geometry.

**Text alternative:** Each scatter point represents one simulated configuration. Marker shape (circles versus squares) and color distinguish heights. Describe the direction of relationships using the correlations printed below; avoid a causal claim.

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for h,color,marker in [(3.5,"#285E91","o"),(7.0,"#A14900","s")]:
    g=df.loc[df.X5==h]
    ax.scatter(g.X1,g.Y1,s=26,alpha=.65,marker=marker,c=color,label=f"{h:g} m; r={g.X1.corr(g.Y1):.3f}")
ax.set(title="Compactness–heating relationships differ by height group",
       xlabel="Relative compactness",ylabel="Heating load (dataset units)")
ax.legend();fig.tight_layout();fig.savefig(OUT/"figure_03_compactness_by_height.png",bbox_inches="tight");plt.show()

### Figure 4. Heating and cooling targets are related
**Text alternative:** Each point compares heating and cooling load for one building configuration. The plot shows their strong positive association; the exact correlation is printed. Marker shape and color identify the height groups.

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for h,color,marker in [(3.5,"#285E91","o"),(7.0,"#A14900","s")]:
    g=df[df.X5==h];ax.scatter(g.Y1,g.Y2,c=color,marker=marker,alpha=.55,s=24,label=f"{h:g} m")
r=df.Y1.corr(df.Y2)
ax.set(title=f"Heating and cooling loads are strongly associated (r = {r:.3f})",
       xlabel="Heating load (dataset units)",ylabel="Cooling load (dataset units)")
ax.legend();fig.tight_layout();fig.savefig(OUT/"figure_04_heating_vs_cooling.png",bbox_inches="tight");plt.show()

## 5. Experimental design and leakage safeguards
**Fixed holdout:** Reserve the same 20% of rows for *both* targets, reproducing the earlier 614/154 random split with `random_state=42`. The test set is used only for final evaluation, never for fitting or hyperparameter selection.

**Training-only model selection:** Use shuffled, five-fold CV on the training subset with a fixed seed. Report CV RMSE mean and standard deviation, then holdout MAE, RMSE and R² for each target separately.

**Feature handling:** Treat X6 (orientation) and X8 (glazing distribution) as categorical codes, one-hot encoded; standardize continuous inputs only within the linear regression pipeline. Tree models use the same one-hot representation for comparable information. Imputation is intentionally absent because missingness must be addressed explicitly in the quality audit.

**Baseline:** Dummy mean prediction and Linear Regression. Models tested: Random Forest (100 trees, as in the original exercise) and Gradient Boosting (default settings). This is a *limited model comparison*, not a comprehensive tuning study. Retain all candidate results rather than reporting only the winner.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[FEATURES].copy(); Y = df[TARGETS].copy()
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=.20,random_state=SEED)
print("Training rows:",len(X_train),"| Final holdout rows:",len(X_test))
assert len(X_train)==614 and len(X_test)==154
assert X_train.index.intersection(X_test.index).empty
CAT=["X6","X8"]; NUM=[c for c in FEATURES if c not in CAT]
# Handle scikit-learn versions accepting sparse or sparse_output.
try: onehot = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
except TypeError: onehot = OneHotEncoder(handle_unknown="ignore",sparse=False)
prep_linear=ColumnTransformer([("continuous",StandardScaler(),NUM),("category",onehot,CAT)])
prep_tree=ColumnTransformer([("continuous","passthrough",NUM),("category",onehot,CAT)])

def candidate_models():
    return {
        "Mean baseline":DummyRegressor(strategy="mean"),
        "Linear regression":Pipeline([("prepare",prep_linear),("model",LinearRegression())]),
        "Random forest":Pipeline([("prepare",prep_tree),("model",RandomForestRegressor(
            n_estimators=100,random_state=SEED,n_jobs=-1))]),
        "Gradient boosting":Pipeline([("prepare",prep_tree),("model",GradientBoostingRegressor(
            random_state=SEED))]),
    }
cv=KFold(n_splits=5,shuffle=True,random_state=SEED)

### Reproduction check: the original heating-load Random Forest exercise
Before the expanded comparisons, repeat **exactly** the earlier modeling choice: eight numeric feature codes (including orientation and glazing distribution), 100 trees, `random_state=42`, and the same 80/20 split. The original reported MAE **0.3546**, RMSE **0.4908**, R² **0.9977**, with a **2.7093** taller/shorter MAE ratio. The cell below measures whether those historical results reproduce; the updated categorical-encoding pipeline may yield different results and should be clearly distinguished.

**Accessibility:** The output is a text table with metrics by height group and an explicit numeric comparison, not a color-coded result.

In [ ]:
# Original experimental specification: use raw coded X6/X8, as in the prior assignment.
legacy_rf = RandomForestRegressor(n_estimators=100, random_state=SEED)
legacy_rf.fit(X_train, Y_train["Y1"])
legacy_pred = legacy_rf.predict(X_test)
legacy_metrics = {
    "MAE":mean_absolute_error(Y_test["Y1"],legacy_pred),
    "RMSE":np.sqrt(mean_squared_error(Y_test["Y1"],legacy_pred)),
    "R2":r2_score(Y_test["Y1"],legacy_pred)
}
print("Original Random Forest reproduction:", {k:round(v,4) for k,v in legacy_metrics.items()})
legacy_rows=[]
for h in [3.5,7.0]:
    m=X_test["X5"].to_numpy()==h
    e=legacy_pred[m]-Y_test["Y1"].to_numpy()[m]
    legacy_rows.append({"height_m":h,"n":int(m.sum()),"MAE":np.abs(e).mean(),
                        "RMSE":np.sqrt(np.mean(e**2)),"mean_signed_error":e.mean()})
legacy_audit=pd.DataFrame(legacy_rows)
display(legacy_audit.round(4))
print("MAE ratio (worst/best):",round(legacy_audit.MAE.max()/legacy_audit.MAE.min(),4))
pd.DataFrame([legacy_metrics]).to_csv(OUT/"table_legacy_random_forest_metrics.csv",index=False)
legacy_audit.to_csv(OUT/"table_legacy_height_reliability.csv",index=False)


## 6. Cross-validation: choose a model on training data only
Run the same candidate algorithms separately for Y1 and Y2. Five-fold CV produces a comparable estimate of training-set generalization and variation across folds. CV performance estimates should not be mistaken for validation on real buildings.

**Accessible output:** The ranked table reports both the average RMSE and its fold-to-fold standard deviation.

In [ ]:
cv_rows=[]
for target in TARGETS:
    for name, estimator in candidate_models().items():
        score=cross_validate(estimator,X_train,Y_train[target],cv=cv,
              scoring={"rmse":"neg_root_mean_squared_error","mae":"neg_mean_absolute_error","r2":"r2"},
              n_jobs=1,error_score="raise")
        cv_rows.append({"target":target,"model":name,
            "cv_rmse_mean":-score["test_rmse"].mean(),"cv_rmse_sd":score["test_rmse"].std(ddof=1),
            "cv_mae_mean":-score["test_mae"].mean(),"cv_r2_mean":score["test_r2"].mean()})
cv_table=pd.DataFrame(cv_rows).sort_values(["target","cv_rmse_mean"]).reset_index(drop=True)
display(cv_table.round(4))
cv_table.to_csv(OUT/"table_04_training_cross_validation.csv",index=False)
selected={t:cv_table.loc[cv_table.target==t].iloc[0]["model"] for t in TARGETS}
print("Training-CV-selected models (do not change based on holdout):",selected)

## 7. Final held-out comparison: heating and cooling
Train each predeclared candidate on the full training partition, then make predictions on the untouched 154-row holdout. This provides apples-to-apples metrics for both outcomes and enables comparison with the original Random Forest result.

**Paper reporting:** Use these holdout rows only for final reporting. If you revise the methodology after examining them, disclose that it is exploratory rather than a fresh, independent test.

In [ ]:
fitted={};predictions={};hold_rows=[]
for target in TARGETS:
    for name,est in candidate_models().items():
        est.fit(X_train,Y_train[target]); pred=est.predict(X_test)
        fitted[(target,name)]=est;predictions[(target,name)]=pred
        hold_rows.append({"target":target,"model":name,
            "test_n":len(pred),"test_mae":mean_absolute_error(Y_test[target],pred),
            "test_rmse":np.sqrt(mean_squared_error(Y_test[target],pred)),
            "test_r2":r2_score(Y_test[target],pred),"selected_by_cv":name==selected[target]})
holdout_table=pd.DataFrame(hold_rows).sort_values(["target","test_rmse"])
display(holdout_table.round(4))
holdout_table.to_csv(OUT/"table_05_final_holdout_models.csv",index=False)
print("Original exercise used unencoded numerical X6/X8 and Y1 random forest, so the updated pipeline may differ.")

### Figure 5. Observed versus predicted load for CV-selected models
Each graph uses one target and the *training-CV-selected* model. The dashed diagonal is ideal agreement; points above it mean overprediction. The same values appear in the saved prediction CSV and metrics table.

**Accessibility:** Plots include axis labels, a diagonal reference label, and an explicit result statement after each figure. No reliance on marker color to identify the comparison.

In [ ]:
prediction_records=[]
for target in TARGETS:
    name=selected[target];actual=Y_test[target].to_numpy();pred=predictions[(target,name)]
    lo=min(actual.min(),pred.min());hi=max(actual.max(),pred.max())
    fig,ax=plt.subplots(figsize=(6.5,5))
    ax.scatter(actual,pred,facecolors="none",edgecolors="#285E91",s=38,linewidth=1.2,label="Test configurations")
    ax.plot([lo,hi],[lo,hi],linestyle="--",color="#8B3C00",label="Perfect agreement")
    ax.set(title=f"Holdout {NAMES[target].lower()}: {name}",
           xlabel=f"Observed {NAMES[target].lower()} (simulated)",
           ylabel=f"Predicted {NAMES[target].lower()}")
    ax.legend();fig.tight_layout()
    fig.savefig(OUT/f"figure_05_{target}_observed_predicted.png",bbox_inches="tight");plt.show()
    errors=pred-actual
    print(f"Accessible summary, {target}, {name}: MAE {np.mean(np.abs(errors)):.3f}, "
          f"RMSE {np.sqrt(np.mean(errors**2)):.3f}, R² {r2_score(actual,pred):.3f}. "
          "Points closest to the dashed diagonal have smallest error.")
    for idx,a,p in zip(X_test.index,actual,pred):
        prediction_records.append({"row_index":idx,"target":target,"model":name,
                                   "observed":a,"predicted":p,"signed_error":p-a,
                                   "absolute_error":abs(p-a),"height":X_test.loc[idx,"X5"]})
prediction_table=pd.DataFrame(prediction_records)
prediction_table.to_csv(OUT/"table_06_selected_model_predictions.csv",index=False)

## 8. Reliability audit: errors by building height
The previous EcoBuild audit showed different prediction errors between 3.5 m and 7.0 m configurations using Random Forest heating-load predictions. Repeat this examination across **all candidate algorithms and both targets**, including sample size, MAE, RMSE, and signed error.

**Terminology:** This is a **model reliability / subgroup performance audit**, *not* a demographic fairness test. Height is a design attribute. Absolute error differences can arise because tall configurations occupy different regions of the feature and target distribution; the observed disparity does not prove bias caused by height.

**Interpretation safeguard:** Report both the absolute MAE gap and the worst/best ratio, with underlying group sample sizes. Ratios can be unstable if the best group's error is near zero.

In [ ]:
rows=[]
for target in TARGETS:
    actual=Y_test[target].to_numpy()
    for name in candidate_models():
        pred=predictions[(target,name)]
        for h in [3.5,7.0]:
            mask=X_test["X5"].to_numpy()==h
            diff=pred[mask]-actual[mask]
            rows.append({"target":target,"model":name,"height_m":h,"n":int(mask.sum()),
                         "MAE":np.mean(np.abs(diff)),"RMSE":np.sqrt(np.mean(diff**2)),
                         "mean_signed_error":np.mean(diff)})
reliability=pd.DataFrame(rows)
display(reliability.round(4))
reliability.to_csv(OUT/"table_07_height_group_reliability.csv",index=False)
gaps=[]
for (target,name), grp in reliability.groupby(["target","model"],sort=False):
    a=grp.set_index("height_m").MAE
    gaps.append({"target":target,"model":name,"MAE_gap":a.max()-a.min(),
                 "MAE_worst_best_ratio":a.max()/a.min() if a.min()>0 else np.nan,
                 "worse_group_m":float(a.idxmax())})
gap_table=pd.DataFrame(gaps)
display(gap_table.round(4))
gap_table.to_csv(OUT/"table_08_height_disparity_summary.csv",index=False)

### Figure 6. Subgroup prediction error (selected model)
Create separate charts for heating and cooling. Exact values and group sizes are printed above, and both groups have explicit labels.

**Text alternative:** Each chart compares mean absolute error for short and tall design groups for one selected model. Lower bars indicate smaller typical prediction errors. The ratio and absolute gap are displayed below; height categories are *not* protected demographic groups.

In [ ]:
for target in TARGETS:
    name=selected[target];g=reliability[(reliability.target==target)&(reliability.model==name)].set_index("height_m").loc[[3.5,7.0]]
    fig,ax=plt.subplots(figsize=(7,4.2))
    bars=ax.bar(["3.5 m (shorter)","7.0 m (taller)"],g.MAE,
                color=["#285E91","#A14900"],width=.55)
    ax.bar_label(bars,labels=[f"{v:.3f} (n={int(n)})" for v,n in zip(g.MAE,g.n)],padding=3)
    ax.set(title=f"Height-group MAE for {NAMES[target].lower()} — {name}",
           xlabel="Building height",ylabel="Mean absolute error (dataset units)")
    ax.set_ylim(0,max(g.MAE)*1.27);fig.tight_layout()
    fig.savefig(OUT/f"figure_06_{target}_reliability.png",bbox_inches="tight");plt.show()
    print(f"Accessible summary, {target}: shorter MAE={g.loc[3.5,'MAE']:.3f}; "
          f"taller MAE={g.loc[7.0,'MAE']:.3f}; worst/best={g.MAE.max()/g.MAE.min():.2f}.")

## 9. Error diagnostics: examine signed residuals
Mean absolute error hides whether the model systematically overpredicts or underpredicts. Signed residuals (`predicted − observed`) help distinguish direction from magnitude. We inspect residuals against observed load, separately for each selected model.

**Text alternative:** Positive errors indicate overprediction; negative errors indicate underprediction. A horizontal zero line marks unbiased point predictions. Inspect clusters, extreme errors, and changes in spread; do not assume visual patterns establish statistical significance.

In [ ]:
for target in TARGETS:
    name=selected[target];actual=Y_test[target].to_numpy();resid=predictions[(target,name)]-actual
    fig,ax=plt.subplots(figsize=(7,4.3))
    ax.scatter(actual,resid,edgecolors="#285E91",facecolors="none",s=34,label="Holdout residuals")
    ax.axhline(0,linestyle="--",color="#963F00",label="Zero residual")
    ax.set(title=f"Residuals versus observed {NAMES[target].lower()} — {name}",
           xlabel=f"Observed {NAMES[target].lower()}",ylabel="Predicted − observed")
    ax.legend();fig.tight_layout()
    fig.savefig(OUT/f"figure_07_{target}_residuals.png",bbox_inches="tight");plt.show()
    print(f"Accessible summary, {target}: mean signed error={resid.mean():.4f}; "
          f"minimum={resid.min():.3f}; maximum={resid.max():.3f}.")

## 10. Permutation importance: interpret predictive dependence
Permutation importance asks how much the model's *holdout-set* prediction error worsens when one input is shuffled. It is **not** a causal effect estimate, and correlated architectural attributes can divide or obscure importance.

**Method:** Use the CV-selected fitted model for each target with five repeated permutations and a fixed seed. Because permutation works on the original input table passed to the model pipeline, each of X1–X8 receives one score even where category encoding is applied internally.

**Accessibility:** Ranked bar plots include feature names on the y-axis; exact means and standard deviations are exported to a table. Negative importance can arise from noise.

In [ ]:
from sklearn.inspection import permutation_importance
importance_rows=[]
for target in TARGETS:
    name=selected[target]; est=fitted[(target,name)]
    p=permutation_importance(est,X_test,Y_test[target],n_repeats=5,random_state=SEED,
                             scoring="neg_root_mean_squared_error",n_jobs=1)
    for f,mean,sd in zip(FEATURES,p.importances_mean,p.importances_std):
        importance_rows.append({"target":target,"model":name,"feature":f,
                                "feature_label":NAMES[f],"rmse_increase_mean":mean,
                                "rmse_increase_sd":sd})
imp=pd.DataFrame(importance_rows)
display(imp.sort_values(["target","rmse_increase_mean"],ascending=[True,False]).round(4))
imp.to_csv(OUT/"table_09_permutation_importance.csv",index=False)
for target in TARGETS:
    d=imp[imp.target==target].sort_values("rmse_increase_mean")
    fig,ax=plt.subplots(figsize=(8,5))
    ax.barh(d.feature_label,d.rmse_increase_mean,xerr=d.rmse_increase_sd,color="#285E91",capsize=3)
    ax.axvline(0,color="#963F00",linewidth=1)
    ax.set(title=f"Permutation importance for {NAMES[target].lower()} (holdout)",
           xlabel="Increase in RMSE after shuffling feature (higher = more dependence)")
    fig.tight_layout();fig.savefig(OUT/f"figure_08_{target}_importance.png",bbox_inches="tight");plt.show()
    print("Accessible summary, top 3 importance scores:")
    print(d.sort_values("rmse_increase_mean",ascending=False).head(3)[["feature_label","rmse_increase_mean"]].to_string(index=False))

## 11. Team-method reference: grouped validation and paper figures (Namita)

**Source notebooks provided by the team:** `EcoBuild_Update3_Grouped_Validation.ipynb` and `EcoBuild_Paper_Figures.ipynb` (Namita, machine-learning/production role). Their code is **not** silently executed or represented as this author's contribution. They are methodological context for comparisons below.

**Confirmed from the uploaded notebooks:**
- The dataset is structured as 192 combinations of seven non-orientation inputs, each simulated for four orientations. A random row split can therefore place a near-duplicate building design on both sides of the evaluation.
- The team's app-oriented model uses engineered terms (wall/roof ratio, envelope/height, glazing×height, two-story indicator), one-hot encoding of orientation and glazing distribution, and Gradient Boosting with separately specified heating/cooling settings.
- Namita's validation notebook examines shuffled row-fold versus whole-design-fold performance, group-held-out tuning and comparisons, and SHAP. The paper-figures notebook goes further by comparing row-based and design-grouped holdouts across **100 repeated splits**, and makes tables and figures for the paper.
- The published *notebook descriptions* mention a larger error under grouped validation, particularly for heating. **Do not copy those numbers into your paper without re-running the notebooks and verifying their underlying outputs.**

**Important alignment distinction:** Sections 5–10 above reproduce **Veronica's original raw-feature Random Forest audit** and extend it to transparent baselines. The following sections reproduce the *team-style validation design* as a clearly labeled secondary analysis. The two pipelines should not be conflated when interpreting metrics.

**Accessibility:** The tables below report all numerical findings; charts are supplemented with plain-language descriptions, data export, readable legends, and direct labeling rather than color alone.


In [ ]:
team_notebook_info = {
    "grouped_validation": "EcoBuild_Update3_Grouped_Validation.ipynb",
    "paper_figures": "EcoBuild_Paper_Figures.ipynb",
    "responsibility": "Namita — team machine learning, grouped validation and SHAP",
    "status": "reviewed source notebooks; not executed as part of this notebook",
    "expected_designs": 192,
    "orientations_per_design": 4,
}
print("Reference notebooks:")
for key, value in team_notebook_info.items(): print(f"  {key}: {value}")


## 12. Secondary sensitivity check: group by geometry only

This check preserves the original notebook's simpler model inputs and asks whether holding out whole **geometries (X1–X5)** changes cross-validation performance. This is *more restrictive* than grouping on all seven non-orientation attributes: it bundles glazing variants of a geometry together. It does not reproduce Namita's exact experiment. Her design definition and a correctly matched experiment follow in Section 13.

**Accessibility:** The printed table identifies the target, model, number of folds, and each evaluation metric. Compare numerical rows rather than colors alone.


In [ ]:
from sklearn.model_selection import GroupKFold
# Group all examples with matching X1–X5 building geometry together within each CV fold.
geometry_cols = ["X1", "X2", "X3", "X4", "X5"]
geom_groups = pd.factorize(pd.MultiIndex.from_frame(X_train[geometry_cols]))[0]
unique_geom = np.unique(geom_groups).size
print("Unique building-geometry groups in training:", unique_geom)
if unique_geom >= 5:
    group_cv = GroupKFold(n_splits=5)
    group_rows = []
    for target in TARGETS:
        for name, estimator in candidate_models().items():
            scores = cross_validate(
                estimator, X_train, Y_train[target],
                cv=group_cv.split(X_train, Y_train[target], groups=geom_groups),
                scoring={"rmse": "neg_root_mean_squared_error",
                         "mae": "neg_mean_absolute_error", "r2": "r2"},
                error_score="raise", n_jobs=1)
            group_rows.append({"target": target, "model": name,
                "geometry_cv_rmse_mean": -scores["test_rmse"].mean(),
                "geometry_cv_rmse_sd": scores["test_rmse"].std(ddof=1),
                "geometry_cv_mae_mean": -scores["test_mae"].mean(),
                "geometry_cv_r2_mean": scores["test_r2"].mean()})
    group_table = pd.DataFrame(group_rows).sort_values(
        ["target", "geometry_cv_rmse_mean"]).reset_index(drop=True)
    display(group_table.round(4))
    group_table.to_csv(OUT / "table_10_geometry_group_validation.csv", index=False)
    print("Interpretation: grouped CV estimates performance on held-out geometry types")
    print("within the simulated dataset, not on real buildings.")
else:
    print("Fewer than five groups; grouped validation skipped.")

## 13. Decision-support example: compare **existing** configurations only
This is a descriptive prototype for an eventual Power BI/Streamlit experience. Use **two observed design rows**, not arbitrary changed inputs. The selected model's prediction helps compare *estimated simulated load* for existing designs; it does not show intervention effects or optimization. It also does not estimate electricity bills.

**Design governance:** Present both estimated loads, mark the data source, warn on out-of-distribution or unsupported inputs, and disclose that the height-group audit shows reliability differences.

In [ ]:
# Example uses actual rows from the holdout set (sorted by observed heating load for illustration).
example_indices=Y_test["Y1"].sort_values().index[[20,-21]]
examples=X_test.loc[example_indices]
scenario=[]
for j,(idx,row) in enumerate(examples.iterrows(),start=1):
    rec={"design":f"Observed test configuration {j}","source_row":int(idx),
         "overall_height_m":row.X5,"relative_compactness":row.X1}
    for target in TARGETS:
        name=selected[target]
        rec[f"estimated_{target}"]=float(fitted[(target,name)].predict(examples.loc[[idx]])[0])
        rec[f"observed_{target}"]=float(Y_test.loc[idx,target])
    scenario.append(rec)
scenario_table=pd.DataFrame(scenario)
display(scenario_table.round(3))
scenario_table.to_csv(OUT/"table_11_example_design_comparison.csv",index=False)
print("These are illustrative comparisons of existing simulated configurations, not recommendations for real construction.")

## 13. Reproduce the team-style experiment: new building designs rather than new orientations

**Purpose.** Our previous 80/20 random-row test provides continuity with the original assignment, but some records share the same geometry, glazing, and building proportions, differing only in orientation. To estimate performance when a *new design* arrives, rows for that same design must remain together during splitting and cross-validation.

**Design ID.** A unique design is defined by `X1, X2, X3, X4, X5, X7, X8` (all inputs except `X6 = Orientation`). This is the definition explicitly used in Namita's notebook. An expected dataset has 192 design IDs, each containing four records. A random row split and a whole-design split answer **different questions**; neither directly evaluates real occupied buildings.

We first verify these properties rather than assume them, then hold out whole designs on a single fixed 80/20 split. This analysis uses the same original baseline candidate models used above, so the **effect of splitting** can be examined without introducing a different preprocessing pipeline.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
DESIGN_COLS = ["X1", "X2", "X3", "X4", "X5", "X7", "X8"]
design_ids = df.groupby(DESIGN_COLS, dropna=False).ngroup()
sizes = pd.Series(design_ids).value_counts().sort_index()
print(f"Records: {len(df)} | Unique designs: {len(sizes)} | Rows per design: {sorted(sizes.unique())}")
if not (len(df) == 768 and len(sizes) == 192 and (sizes == 4).all()):
    print("WARNING: Dataset does not match the 192-by-4 grouping assumption. Check data source and design definition.")
group_holdout = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
gtrain, gtest = next(group_holdout.split(X, groups=design_ids))
assert not (set(design_ids.iloc[gtrain]) & set(design_ids.iloc[gtest]))
print("Whole-design split: train",len(gtrain),"rows /",len(set(design_ids.iloc[gtrain])),"designs; test",len(gtest),"rows /",len(set(design_ids.iloc[gtest])),"designs")
comparison=[]
for target in TARGETS:
    for model_name, estimator in candidate_models().items():
        estimator.fit(X.iloc[gtrain], Y[target].iloc[gtrain])
        predicted = estimator.predict(X.iloc[gtest])
        comparison.append({"target":target,"model":model_name,"holdout":"Whole designs (fixed)","MAE":mean_absolute_error(Y[target].iloc[gtest],predicted),"RMSE":np.sqrt(mean_squared_error(Y[target].iloc[gtest],predicted)),"R2":r2_score(Y[target].iloc[gtest],predicted)})
        original=holdout_table[(holdout_table.target==target)&(holdout_table.model==model_name)].iloc[0]
        comparison.append({"target":target,"model":model_name,"holdout":"Random rows (fixed)","MAE":original.test_mae,"RMSE":original.test_rmse,"R2":original.test_r2})
design_comparison=pd.DataFrame(comparison)
display(design_comparison.round(4).sort_values(["target","model","holdout"]))
design_comparison.to_csv(OUT/"table_11_row_vs_design_fixed_holdouts.csv",index=False)
print("Note: Different random 80/20 holdouts are not a controlled paired comparison; repeated splitting below reduces dependence on one seed.")


## 14. Repeated split comparison: how much does evaluation design change the reported error?

**Purpose.** A single split can be unusually easy or difficult. We repeat 80/20 row-based and design-grouped holdouts over several independent random seeds, train identical models on each split, and calculate the distribution of out-of-sample errors for both targets. This mirrors the central research question in Namita's paper-figures notebook, while keeping Veronica's **raw-feature baseline pipeline** unchanged.

**Computation.** `REPEATS = 20` is a convenient exploratory default. Set it to `100` when preparing final paper tables to match Namita's repeated-design study. The repeated runs do **not** constitute 100 independent datasets and do not directly establish causal or field-deployment performance. Because splits overlap, treat empirical intervals as *descriptive split variability*, not formal independent confidence intervals. Model hyperparameters are fixed in advance; no test-set tuning occurs.

**Accessibility:** The exported long-format CSV includes one row per target, model, split strategy and repetition. Numeric summary columns show mean, standard deviation, and the 2.5th/97.5th percentiles of error. No interpretation requires color.


In [ ]:
REPEATS = 20  # Change to 100 for final research-paper comparisons.
REPEATED_MODELS = ["Mean baseline", "Linear regression", "Random forest", "Gradient boosting"]
repeated_rows=[]
for rep in range(REPEATS):
    random_split = train_test_split(np.arange(len(X)), test_size=.2, random_state=SEED+rep)
    group_split = next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED+rep).split(X, groups=design_ids))
    for split_label, (tr, te) in [("Random rows",random_split),("Whole designs",group_split)]:
        for target in TARGETS:
            for model_name in REPEATED_MODELS:
                estimator = candidate_models()[model_name]
                estimator.fit(X.iloc[tr], Y[target].iloc[tr])
                pred = estimator.predict(X.iloc[te])
                repeated_rows.append({"repetition":rep,"seed":SEED+rep,"strategy":split_label,
                    "target":target,"model":model_name,"train_rows":len(tr),"test_rows":len(te),
                    "test_designs":len(set(design_ids.iloc[te])),
                    "MAE":mean_absolute_error(Y[target].iloc[te],pred),
                    "RMSE":np.sqrt(mean_squared_error(Y[target].iloc[te],pred)),
                    "R2":r2_score(Y[target].iloc[te],pred)})
repeated_results=pd.DataFrame(repeated_rows)
repeated_results.to_csv(OUT/"table_12_repeated_split_raw.csv",index=False)
repeated_summary=repeated_results.groupby(["target","model","strategy"],as_index=False).agg(
    repeats=("MAE","size"), mean_MAE=("MAE","mean"),sd_MAE=("MAE","std"),
    p025_MAE=("MAE",lambda a:np.percentile(a,2.5)),p975_MAE=("MAE",lambda a:np.percentile(a,97.5)),
    mean_RMSE=("RMSE","mean"),mean_R2=("R2","mean"))
repeated_summary.to_csv(OUT/"table_13_repeated_split_summary.csv",index=False)
display(repeated_summary.round(4))


### Figure 9. Random-row versus whole-design MAE across repeated holdouts

**Chart guidance.** Show two explicit category labels, use distinct markers, retain the same MAE scale for each target, and place a numerical table next to the figure. A higher MAE for whole-design testing suggests the random-row estimate is optimistic for genuinely new combinations of building attributes. It does **not** prove deployment accuracy on actual buildings.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5),sharey=False)
for ax,t in zip(axes,TARGETS):
    for name,mark in [("Linear regression","o"),("Random forest","s"),("Gradient boosting","^")]:
        sub=repeated_summary[(repeated_summary.target==t)&(repeated_summary.model==name)]
        a=float(sub[sub.strategy=="Random rows"].mean_MAE.iloc[0])
        b=float(sub[sub.strategy=="Whole designs"].mean_MAE.iloc[0])
        ax.plot([0,1],[a,b],marker=mark,label=name,linewidth=1.6)
        ax.annotate(f"{a:.2f}",(0,a),xytext=(-9,5),textcoords="offset points",fontsize=8)
        ax.annotate(f"{b:.2f}",(1,b),xytext=(4,5),textcoords="offset points",fontsize=8)
    ax.set_xticks([0,1],["Random rows","Whole designs"])
    ax.set_xlim(-.18,1.22);ax.set_ylabel("Mean test MAE (load units; lower is better)")
    ax.set_title(f"{NAMES[t]} — {REPEATS} repeats")
    ax.grid(axis="y",alpha=.25)
axes[0].legend(loc="best",fontsize=9)
fig.suptitle("Mean prediction error changes when entire building designs are withheld",fontsize=13)
fig.tight_layout()
fig.savefig(OUT/"figure_09_repeated_split_comparison.png",bbox_inches="tight",dpi=300)
plt.show()
print("Accessible alternative: see table_13_repeated_split_summary.csv for all exact plotted values.")


## 15. Evaluate subgroup reliability on whole-design holdouts

The original height audit used a random-row split of a raw Random Forest and reported a **2.7093×** MAE ratio between taller and shorter designs. We retain that result for continuity. Here we ask whether the same pattern appears when the model is evaluated on **held-out designs**, for each target and model. Differences in group error may reflect target scale, sample composition, and correlated design features; no protected demographic groups exist in this dataset.

**Accessibility:** The results table explicitly includes height group, sample count, MAE, RMSE, and mean signed error (predicted minus actual). Signed values below zero indicate average underprediction. We compute within-model higher/lower MAE ratios only when both groups exist and have positive denominator.


In [ ]:
height_audit=[]
for target in TARGETS:
    for model_name, estimator in candidate_models().items():
        estimator.fit(X.iloc[gtrain],Y[target].iloc[gtrain]); pred=estimator.predict(X.iloc[gtest])
        for h in sorted(X.iloc[gtest].X5.unique()):
            mask=(X.iloc[gtest].X5.to_numpy()==h)
            actual=Y[target].iloc[gtest].to_numpy()[mask]
            err=pred[mask]-actual
            height_audit.append({"target":target,"model":model_name,"height_m":h,"n":len(err),
                "MAE":np.abs(err).mean(),"RMSE":np.sqrt(np.mean(err**2)),"mean_signed_error":err.mean()})
height_audit_grouped=pd.DataFrame(height_audit)
display(height_audit_grouped.round(4))
height_audit_grouped.to_csv(OUT/"table_14_grouped_holdout_height_audit.csv",index=False)
ratio_table=(height_audit_grouped.groupby(["target","model"]).MAE
    .agg(lambda a:float(a.max()/a.min()) if len(a)==2 and a.min()>0 else np.nan)
    .rename("worst_to_best_MAE_ratio").reset_index())
display(ratio_table.round(3))
ratio_table.to_csv(OUT/"table_15_grouped_holdout_height_disparity.csv",index=False)


## 16. Reading Namita's model-specific findings alongside this independent analysis

The team notebooks also include app-specific feature engineering, hyperparameter optimization, SHAP and full figure generation. Those are **Namita's contributions**, not evidence that this notebook independently implemented her system. Use them in the IEEE paper with explicit team attribution and with metrics taken from executed outputs, not from text captions alone.

| Question | Evidence source | Responsible interpretation |
|---|---|---|
| Why was the 80/20 random split potentially optimistic? | Namita's grouped-validation notebook; our Sections 13–14 | Related rows from the same design can cross split boundaries |
| Does held-out-design evaluation change error? | Our repeated comparison and Namita's 100-repeat figures | Discuss model/target dependence and variability |
| Which model was selected for the final app? | Namita's model pipelines, validated team repository metadata | Do not assume our baseline winner equals the deployed model |
| Which inputs influence a prediction? | Namita's SHAP outputs and our permutation importance | These describe model dependence, not physical causal effects |
| How useful is this for a design decision? | Veronica's Power BI EDA and reliability audit | Explain tradeoffs, subgroup uncertainty, and limits of synthetic data |

### Accessibility and research ethics checklist
- Every quantitative figure has a distinct descriptive title, axis units, readable labels, and a corresponding CSV table.
- In the paper, write textual alt descriptions for exported figures and explain what patterns matter; do not use the color alone to identify groups.
- Describe the ENB2012 targets as **simulated heating/cooling loads**, not meter readings or observed cost savings.
- Describe subgroup differences as **performance consistency**, not demographic fairness or evidence of discriminatory treatment.
- Differentiate a validation *sensitivity analysis* from the primary deployed-model evaluation.
- Credit Namita for team ML validation and SHAP, and Anthony for verified implementation contributions; name Veronica's independently authored analytics and audit work.


## 17. Export methods metadata, figures, and paper-ready tables


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
meta={
    "project":"EcoBuild", "author":"Veronica Broadway", "dataset_source":DATA_SOURCE,
    "dataset_rows":int(len(df)), "columns":FEATURES+TARGETS,
    "random_seed":SEED,"train_test_split":"80/20 by rows, sklearn random_state=42",
    "training_rows":len(X_train),"holdout_rows":len(X_test),
    "cv":"5-fold shuffled KFold on training data only", "model_selection_metric":"CV RMSE",
    "candidate_models":list(candidate_models()),"selected_by_training_cv":selected,
    "teammate_reference":team_notebook_info,
    "design_grouping": "All seven non-orientation inputs X1-X5,X7,X8; expected 192 designs x 4 orientations",
    "design_grouped_validation": "Fixed GroupShuffleSplit 20%, plus repeated GroupShuffleSplit vs random-row",
    "repeated_split_n": REPEATS,
    "categorical_variables":CAT,"continuous_variables":NUM,"software":VERSIONS,
    "scope_limitations":["Simulation-generated targets, not real-building measurements",
        "No claims of causal effects or actual energy savings", "Height-group audit is not demographic fairness",
        "No external dataset validation", "No hyperparameter tuning beyond predeclared defaults"],
    "notes":"Read figure descriptions in notebook; figures do not include embedded alt text."
}
with open(OUT/"methods_metadata.json","w",encoding="utf-8") as f:json.dump(meta,f,indent=2)
archive=Path("EcoBuild_Research_Results.zip")
with ZipFile(archive,"w",ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):z.write(p,arcname=p.name)
print("Saved outputs:")
for p in sorted(OUT.iterdir()):print("-",p.name)
print("ZIP:",archive.resolve())

## 18. Paper integration guide and remaining evidence

| IEEE section | Primary outputs / evidence | Key limitation to state |
|---|---|---|
| Introduction | Research question and EDA motivation | Simulation-based, not field validated |
| Related Work | Scholarly comparison and dataset provenance (add verified citations in Overleaf) | This notebook does not replace literature synthesis |
| Data | Data dictionary, `table_01`–`table_03`, Figures 1–4 | Fixed synthetic experiment conditions |
| Methodology | Split, categorical encoding, baselines, CV and tests | Method choices must match team implementation |
| Results | `table_04`–`table_10`, Figures 5–8 | Small test sample, comparisons exploratory |
| Discussion | Reliability disparities, interpretability, design comparison | Different configuration groups; no causation, no demographic fairness |
| Reproducibility | JSON, CSVs, figures, ZIP, notebook | Record exact dataset version / source |

### Attribution and boundaries
- **Veronica's documented work:** EDA and Power BI visualization, project/analytics and governance work, initial heating Random Forest and building-height reliability audit. Newly added comparisons run here should be represented as additional analysis **only after they are actually executed and reviewed**.
- **Namita:** Team machine-learning and SHAP work should be credited accurately; do not describe her model artifacts as independently implemented here.
- **Anthony:** Credit broader project implementation, subject to confirmation of exact tasks.

### Items to obtain before final claims
- [ ] Confirm whether the team adopted the same final model, preprocessing, and evaluation protocol.
- [ ] Obtain Namita's actual SHAP plots and describe them with proper credit.
- [ ] Obtain Anthony's deployment/interface details and project repository URL.
- [ ] Decide which generated figures fit the 6–8 page IEEE conference limit.
- [ ] Incorporate verified scholarly sources (10+ for Related Work) and professor's Nov. 8 feedback when available.
- [ ] Provide an AI-use statement and link to replicable code/data.

### Conclusion for the notebook
This analysis is designed to distinguish **high average accuracy** from **consistent, interpretable performance** across simulated building designs. All numerical conclusions should be written after the notebook has executed and outputs have been checked; the code does not validate real-building transfer or produce operational energy savings.